<a href="https://colab.research.google.com/github/Jupiterian/ML-Learning-Portfolio/blob/main/Linear%20Algebra/Gram_Schmidt.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gram-Schmidt
The following notebook is a custom class to perform Gram-Schmidt operations on a matrix.
Gram-Schmidt takes a linearly independent set of vectors and produces a set of orthnormal vectors. In this context, the algorithm takes in the column space of a matrix A, and returns an orthonormal matrix $Q$, as well as the upper-triangular matrix $R$ such that
$$A=QR$$.

## Code Explanation
Given a matrix $A$, the Gram-Schmidt algorithm returns an **orthonormalized** matrix $Q$ where the columns of Q are orthogonal to one another and they are normalized, having a magnitude of 1.

Thus the entire Gram-Schmidt process can be broken into two separate parts: orthogonalization (the hard part), and normalization (the easy part).

### Gram-Schmidt Process
Normalization: When I say a vector is normalized, I mean that the vector is divided by its magnitude. For example, normalizing the vector $\begin{bmatrix} 3 \\ 4 \end{bmatrix}$ can be performed by dividing the vector by its magnitude, $5$ to get the vector $\begin{bmatrix} \frac{3}{5} \\ \frac{4}{5} \end{bmatrix}$.
<br/>
Suppose we have an $m$ x $n$ matrix A with columns $v_1, v_2, ... v_n$. We want to produce orthnormal columns of a matrix denoted $Q$ represented as $q_1, q_2, ... q_n$.
$v_1$ becomes the first direction in the orthogonal direction. This column is merely normalized to produce $q_1$.
Remaining orthogonal vectors are calculated by computing the error when projecting themselves onto the existing vectors. Recall that when projecting a matrix $b$ onto $a$, the error vector $b-ax$ is perpendicular to $a$.
This process repeats. $v_2$ is orthogonalized by being projected onto $q_1$, taking the error, and then normalizing it to produce $q_2$. $v_3$ is orthogonalized by being projected onto $q_1$, and then taking the error, and then subtracting the error from its projection onto $q_2$ as well.

Here is a general formula for orthogonalizing the vector $v_n$ with respect to vectors $q_1 ... q_{n-1}$.
$$v_n - \sum_{i=1}^{n-1} q_i^{T}v_n q_i$$.

This comes directly from the projection formula for calculating projection $p=ax$ for projecting the vector $b$ onto $a$, however, in this case because $q$ is normal the dot product of $q$ ($q^T q$) with itself is just 1.

#### Side-Note: Calculating R
One of the nice properties of our orthonormal matrix $Q$ is that $Q^TQ = I$ and by extension $Q^T = Q^{-1}$.
$$\begin{bmatrix}  - & q_1^T & - \\ - & q_2^T & - \\ & \vdots & \\ - & q_n^T & - \end{bmatrix} \begin{bmatrix} | & | & & | \\ q_1 & q_2 & \cdots & q_n \\ | & | & & | \end{bmatrix} = \begin{bmatrix} 1 & 0 & ... & 0 \\ 0 & 1 & ... & 0 \\ \vdots & \vdots &  & \vdots
\\ 0 & 0 & ... & 1 \end{bmatrix}$$.
As you can see for all $q_i^Tq_j$ if $i=j$ then the dot product is equal to 1, but if $i \neq j$ then $q_i^Tq_j$ = 0.
Thus, we get
$$A = QR$$
$$R = Q^TA$$

In [2]:
import numpy as np
import math

In [28]:
class QR:
  def __init__(self, A):
    self.A = A

  def normalize(self, vector):
    # Normalize a vector. This will be used in the overall process
    ## standard python code
    '''
    sqmag = 0
    for x in vector:
      sqmag+= (x**2)
    mag = math.sqrt(sqmag)
    normvector = 1/mag * vector
    return normvector
    '''

    ## numpy code
    mag = np.linalg.norm(vector)
    normvector = 1/mag * vector
    return normvector

  def Qcreator(self):
    # apply gram-scmidt to get Q


    ## step 1: normalize 1st vector
    Q = np.array(self.normalize((self.A[:,[0]])))
    ## step 2: orthogonalize
    colIndex = 1

    while colIndex < len(self.A[1]):
      colToOrth = self.A[:, [colIndex]]
      ## selectively subtract column multiples from Q
      for i in range(0, Q.shape[1]):
        colToOrth = colToOrth - (np.dot((Q[: , [i]]).T, colToOrth) * Q[:, [i]])
      Q = np.column_stack((Q, self.normalize(colToOrth)))
      colIndex+=1
    return Q
  def Rcreator(self):
    ## for the test
    R = (self.Qcreator()).T @ self.A
    return R

  def orthonormalize(self):
    return((self.Qcreator(), self.Rcreator()))


In [29]:
## comparison of results np.linalg.qr vs my class
rng = np.random.default_rng(seed=42)

arr = rng.random((2,2))
cool = QR(arr)
print(cool.orthonormalize())
print(np.linalg.qr(arr))

(array([[ 0.66954664, -0.74277002],
       [ 0.74277002,  0.66954664]]), array([[ 1.15594046e+00,  8.11833648e-01],
       [-5.17469038e-16,  1.40934671e-01]]))
QRResult(Q=array([[-0.66954664, -0.74277002],
       [-0.74277002,  0.66954664]]), R=array([[-1.15594046, -0.81183365],
       [ 0.        ,  0.14093467]]))
